We continue our demonstration of the full uncertainty quantification workflow with Jezebel. Here we perform the setup needed to run OpenMC with the perturbed cross sections

In [11]:
#Import modules
import WINDIGO
import os
from pathlib import Path
import openmc

In [2]:
#Save the paths to the perturbed cross section libraries from the previous example
pu239_direct_perturbation_ace_files = ('/mnt/c/Users/jacob/frendy_20241030/Pu239_DirectPerturbationACEFiles_ReactionMT_18')
pu239_random_sampling_ace_files = ('/mnt/c/Users/jacob/frendy_20241030/Pu239_RandomSamplingACEFiles_ReactionMT_18')

To utilize our perturbed cross sections, we need to first reformat them into the OpenMC-compatible .h5 format, and then construct custom cross_sections.xml files using the perturbed files. WINDIGO has a function that interfaces with OpenMC to perform both of these functions. To do this, we need knowledge of the specific isotopes and thermal scattering law data that will remain unperturbed within our model of interest. For Jezebel, we will neglect thermal scattering data.

The specific model for Jezebel is taken from a publically available GitHub repository from MIT's Computation Reactor Physics Group: https://github.com/mit-crpg/benchmarks/blob/master/icsbep/pu-met-fast-001/mcnp/input

In [5]:
#Set the list of nuclides whose data will remain unperturbed
unperturbed_nuclides = ['Pu240', 'Pu241', 'Ga69', 'Ga71']

#Create the perturbed cross_sections.xml files for the direct perturbation ACE files
direct_perturbation_models_directory = WINDIGO.build_perturbed_cross_sections_libraries(
                                       unperturbed_nuclide_list = unperturbed_nuclides,
                                       neutron_sublibrary_path = '/mnt/c/Users/jacob/endfb80/endfb-viii.0-hdf5/neutron',
                                       perturbed_ACE_folder_path = pu239_direct_perturbation_ace_files,
                                       perturbed_nuclide = 'Pu239',
                                       model_name='Jezebel',
                                       perturbation_type='Direct'
)

#Create the perturbed cross_sections.xml files for the random sampling ACE files
random_sampling_models_directory = WINDIGO.build_perturbed_cross_sections_libraries(
                                   unperturbed_nuclide_list = unperturbed_nuclides,
                                   neutron_sublibrary_path = '/mnt/c/Users/jacob/endfb80/endfb-viii.0-hdf5/neutron',
                                   perturbed_ACE_folder_path = pu239_random_sampling_ace_files,
                                   perturbed_nuclide = 'Pu239',
                                   model_name='Jezebel',
                                   perturbation_type='Random'
)

No thermal scattering data will be included.
All perturbed cross_sections.xml files created. The folders containing them are located in: Jezebel_Pu239_DirectPerturbedModels
No thermal scattering data will be included.
All perturbed cross_sections.xml files created. The folders containing them are located in: Jezebel_Pu239_RandomPerturbedModels


When now have a set of folders with perturbed cross_sections.xml files that we can use to run our model with the modified data. To do this, we will need to enter into each directory and create the xml files for our materials, geometry, and settings. 

Let's define the functions to create our models.

In [47]:
def createModel(xs_directory):
    """
    Creates the materials.xml file needed to run the Jezebel model
    
    Parameters
    -------------

    xs_directory: "str: or "os.Path_like object"
        Directory that points to where the cross_sections.xml file is

    Returns
    -------------
    None
    
    """
    
    pu_metal = openmc.Material()
    pu_metal.add_nuclide('Pu239', 0.037047, 'ao')
    pu_metal.add_nuclide('Pu240', 0.0017512, 'ao')
    pu_metal.add_nuclide('Pu241', 0.00011674, 'ao')
    pu_metal.add_nuclide('Ga69', 8.26605216e-4, 'ao')
    pu_metal.add_nuclide('Ga71', 5.48594784e-4, 'ao')
    pu_metal.set_density('atom/b-cm', 0.04029014)
    
    materials = openmc.Materials([pu_metal,])

    materials.cross_sections = xs_directory
    
    materials.export_to_xml()

    #Create the geometry and export it to an xml file
    sphere_boundary = openmc.Sphere(r = 6.3849, boundary_type = 'vacuum')
    sphere_region = -sphere_boundary
    sphere_cell = openmc.Cell()
    sphere_cell.region = sphere_region
    sphere_cell.fill = pu_metal
    
    root_universe = openmc.Universe(cells = (sphere_cell,))
    
    geometry = openmc.Geometry(root_universe)
    geometry.export_to_xml()
    
def createSettings(temperature):
    """
    Creates the settings.xml file needed to run the Jezebel model
    
    Parameters
    -------------

    temperature: "int"
        Value that determines the default temperature of the model

    Returns
    -------------
    None
    
    """

    settings = openmc.Settings()

    bounds = [-7, -7, -7, 7, 7, 7]
    uniform_dist = openmc.stats.Box(bounds[:3], bounds[3:], only_fissionable = True)
    settings.source = openmc.IndependentSource(space = uniform_dist)
    
    settings.batches = 1000
    settings.inactive = 50
    settings.particles = 500000
    
    settings.temperature = {'default': temperature}
    settings.temperature['method'] = 'interpolation'
    
    settings.export_to_xml()

We then iterate through each set of folders containing our cross_sections.xml files, and use these functions to create models of Jezebel that will use our perturbed cross section data.

In [50]:
#Set the current working directory to the examples folder

os.chdir('examples')

#Save the starting directory

starting_directory = os.getcwd()

#Iterate through the direct perturbation folders and create the models

direct_perturbation_path = Path(direct_perturbation_models_directory)

for folder in direct_perturbation_path.iterdir():
    os.chdir(folder)
    createModel('cross_sections.xml')
    createSettings(temperature=300)
    os.chdir(starting_directory) 

#Iterate through the random sampling folders and create the models

random_sampling_path = Path(random_sampling_models_directory)

for folder in random_sampling_path.iterdir():
    os.chdir(folder)
    createModel('cross_sections.xml')
    createSettings(temperature=300)
    os.chdir(starting_directory)

/home/jacob/miniconda3/envs/openmc-env/lib/python3.13/site-packages/openmc/stats/multivariate.py:960: FutureWarning: The 'only_fissionable' has been deprecated. Use the 'constraints' argument when defining a source instead.
  warn("The 'only_fissionable' has been deprecated. Use the "


We are now able to run our models of Jezebel to get the k_eff values needed to calculate the propagated uncertainty.

In [ ]:
#Iterate through the direct perturbation folders and run OpenMC

counter = 0

for folder in direct_perturbation_path.iterdir():
    os.chdir(folder)
    openmc.run(output=False)
    counter += 1
    os.chdir(starting_directory)

#Iterate through the random sampling folders and run OpenMC

counter = 0

for folder in random_sampling_path.iterdir():
    os.chdir(folder)
    openmc.run(output=False)
    counter += 1
    os.chdir(starting_directory)

We need one more piece of simulation data before we can proceed to our post-processing. We have to get an unperturbed k_eff value that will be used as part of the direct perturbation methodology.


In [55]:
#Set up the base model with unperturbed cross sections
os.mkdir('BaseModelFolder')
os.chdir('./BaseModelFolder')

xs_xml_path = '/mnt/c/Users/jacob/endfb80/endfb-viii.0-hdf5/cross_sections.xml'

createModel(xs_xml_path)
createSettings(temperature=300)

#Run the base model
openmc.run(output=False)

print('Base Model Run Completed')

/home/jacob/miniconda3/envs/openmc-env/lib/python3.13/site-packages/openmc/stats/multivariate.py:960: FutureWarning: The 'only_fissionable' has been deprecated. Use the 'constraints' argument when defining a source instead.
  warn("The 'only_fissionable' has been deprecated. Use the "


Base Model Run Completed


We now have completed all of the OpenMC runs required for our analysis.